# 🦷 Oral Disease Classification — Training Final

**Fixes & improvements over V1:**
- ✅ Clean stratified split on **original** images only — no augmented-data leakage
- ✅ **Focal Loss** (γ=2) + Label Smoothing (0.1) — tackles Calculus/Gingivitis confusion
- ✅ **BatchNormalization always frozen** during fine-tuning (protects pretrained stats)
- ✅ Fast evaluation: `model.predict(dataset)` once, not per-batch loop
- ✅ Real inference timing: `model(img, training=False)` with warm-up
- ✅ `get_augmentation()` — each model gets its own fresh instance
- ✅ SpatialDropout2D in CNN conv blocks
- ✅ XLA JIT compilation (~10-20% speed boost)
- ✅ `class_weight` compensates for small Mouth Ulcer / Tooth Discoloration sets

In [ ]:
!pip install -q scikit-learn matplotlib seaborn


In [ ]:
import os, time, warnings, tempfile, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             classification_report, f1_score)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import losses, callbacks, layers, optimizers, regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, GlobalAveragePooling2D,
                                     SpatialDropout2D, Dropout, BatchNormalization,
                                     Input, Dense)
from tensorflow.keras.applications import (
    EfficientNetB0, ResNet50, DenseNet121,
    efficientnet, resnet, densenet
)

warnings.filterwarnings('ignore')
print(f"TensorFlow : {tf.__version__}")
print(f"GPUs       : {tf.config.list_physical_devices('GPU')}")


In [ ]:
# ── Reproducibility ────────────────────────────────────────
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ── Mixed Precision: FP16 on T4 Tensor Cores ───────────────
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f"Mixed Precision: {tf.keras.mixed_precision.global_policy().name}")

# ── XLA JIT compilation (~10-20% extra speed) ──────────────
tf.config.optimizer.set_jit(True)
print("Enabling XLA for faster training")

# ── Multi-GPU Strategy (T4 x2 on Kaggle) ───────────────────
strategy = tf.distribute.MirroredStrategy()
NUM_GPUS  = strategy.num_replicas_in_sync
print(f"GPUs in use: {NUM_GPUS}")

# ── Hyperparameters ─────────────────────────────────────────
IMG_SIZE        = 224
BATCH_SIZE      = 64 * NUM_GPUS   # 128 on Kaggle T4x2
PHASE1_EPOCHS   = 15
FINETUNE_EPOCHS = 200
EPOCHS          = 200
TEST_LIST_PATH  = 'test_image_list.json'
IMG_FORMATS     = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

print(f"Batch size : {BATCH_SIZE}")
print(f"Epochs     : CNN={EPOCHS} | Phase1={PHASE1_EPOCHS} | Finetune={FINETUNE_EPOCHS}")


## 📂 Smart Data Collection

> **Key change from V1:** We collect **original images only**.
> Pre-augmented folders (`preview/`, `augmented/`) are skipped.
> Mixing them into val/test causes **data leakage** — the model trains on
> augmented versions of its own test images, inflating reported accuracy.
> Augmentation is handled **inside the model** during training only.

In [ ]:
# ── Detect main source path (Kaggle or local) ──────────────
for candidate in ['/kaggle/input/datasets/My Oral Diseases Dataset',
                  '/kaggle/input/oral-diseases',
                  'Data']:
    if os.path.exists(candidate):
        SOURCE = candidate
        break
else:
    SOURCE = '.'
print(f"Main source : {SOURCE}")

# ── Detect YOLO dataset (optional, adds Gingivitis + Caries) ─
YOLO_CANDIDATES = [
    '/kaggle/input/caries-gingivitus-toothdiscoloration-ulcer',
    '/kaggle/input/oral-diseases-yolo',
    'Data/Caries_Gingivitus_ToothDiscoloration_Ulcer-yolo_annotated-Dataset',
]
YOLO_BASE = None
for _c in YOLO_CANDIDATES:
    _sub = os.path.join(_c, 'Caries_Gingivitus_ToothDiscoloration_Ulcer-yolo_annotated-Dataset',
                        'Data', 'images')
    if os.path.exists(_sub):
        YOLO_BASE = os.path.join(_c,
                                 'Caries_Gingivitus_ToothDiscoloration_Ulcer-yolo_annotated-Dataset',
                                 'Data')
        break
    # also try direct path structure
    _sub2 = os.path.join(_c, 'Data', 'images')
    if os.path.exists(_sub2):
        YOLO_BASE = os.path.join(_c, 'Data')
        break
print(f"YOLO source : {YOLO_BASE if YOLO_BASE else 'Not found — YOLO extras skipped'}")


In [ ]:
# ── Class names (order must match for all datasets) ─────────
CLASS_NAMES = ['Calculus', 'Data caries', 'Gingivitis',
               'Mouth Ulcer', 'Tooth Discoloration', 'hypodontia']

# Substrings that identify augmented-only folders to EXCLUDE
EXCLUDE_KEYWORDS = [
    'augmented', 'preview',
    'caries_gingivitus',   # whole YOLO dataset dir
    'yolo_annotated',
]

def is_excluded(path: str) -> bool:
    p = path.lower().replace('\\', '/')
    return any(kw in p for kw in EXCLUDE_KEYWORDS)


def collect_originals_for_class(cls_name: str):
    """Function to read images from my dataset folders."""
    paths = []
    cls_key = cls_name.lower().replace(' ', '').replace('_', '')
    for root, _, files in os.walk(SOURCE):
        if is_excluded(root):
            continue
        root_key = root.lower().replace(chr(92), '/').replace(' ', '').replace('_', '')
        match_key = 'ulcer' if cls_name == 'Mouth Ulcer' else cls_key
        if match_key in root_key:
            for f in files:
                if os.path.splitext(f)[1].lower() in IMG_FORMATS:
                    paths.append(os.path.join(root, f))
    return paths


def collect_yolo_extras():
    """
    Extract single-label Gingivitis (class 3) and Caries (class 0)
    images from the YOLO dataset.
    Only single-label images are used — if a photo has >1 class
    annotated it is ambiguous and excluded.
    """
    # YOLO class mapping: 0=Caries, 1=Ulcer, 2=Tooth Discoloration, 3=Gingivitis
    YOLO_USE = {0: 'Data caries', 3: 'Gingivitis'}
    extras = {'Data caries': [], 'Gingivitis': []}
    if YOLO_BASE is None:
        return extras
    for split in ['train', 'val']:
        img_dir = os.path.join(YOLO_BASE, 'images', split)
        lbl_dir = os.path.join(YOLO_BASE, 'labels', split)
        if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
            continue
        for lbl_file in os.listdir(lbl_dir):
            if not lbl_file.endswith('.txt'):
                continue
            with open(os.path.join(lbl_dir, lbl_file)) as f:
                lines = [l.strip() for l in f if l.strip()]
            try:
                classes_in_img = set(int(l.split()[0]) for l in lines)
            except Exception:
                continue
            if len(classes_in_img) == 1:
                cls_id = next(iter(classes_in_img))
                if cls_id in YOLO_USE:
                    stem = os.path.splitext(lbl_file)[0]
                    for ext in ['.jpg', '.jpeg', '.png']:
                        img_path = os.path.join(img_dir, stem + ext)
                        if os.path.exists(img_path):
                            extras[YOLO_USE[cls_id]].append(img_path)
                            break
    return extras


# ── Gather all paths + integer labels ───────────────────────
yolo_extras = collect_yolo_extras()
all_paths, all_labels = [], []
class_counts_orig = {}

print('\n=== Loading my original dataset images ===')
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    paths = collect_originals_for_class(cls_name)
    extra = yolo_extras.get(cls_name, [])
    paths = paths + extra
    class_counts_orig[cls_name] = len(paths)
    all_paths.extend(paths)
    all_labels.extend([cls_idx] * len(paths))
    note = f' (+{len(extra)} YOLO)' if extra else ''
    print(f'  {cls_name:<25}: {len(paths):>5} images{note}')

print(f'\nTotal: {len(all_paths)} images across {len(CLASS_NAMES)} classes')
print('\n✅ Only original images collected — no pre-augmented files!')


In [ ]:
# ── Visualize class distribution ────────────────────────────
COLORS = ['#e94560','#4ecdc4','#45b7d1','#f7dc6f','#a29bfe','#fd79a8']
total  = sum(class_counts_orig.values())

print(f"{'Class':<25} {'Count':>6} {'%':>8}")
print('─' * 42)
for cls in CLASS_NAMES:
    cnt = class_counts_orig[cls]
    print(f"{cls:<25} {cnt:>6} {cnt/total*100:>7.1f}%")
print('─' * 42)
print(f"{'Total':<25} {total:>6}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
counts_list = [class_counts_orig[c] for c in CLASS_NAMES]

bars = ax1.bar(CLASS_NAMES, counts_list, color=COLORS, edgecolor='white')
ax1.set_title('Original Images per Class (No Data Leakage)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Count'); ax1.tick_params(axis='x', rotation=25)
for bar, v in zip(bars, counts_list):
    ax1.text(bar.get_x()+bar.get_width()/2, v+5, str(v), ha='center', fontweight='bold')

ax2.pie(counts_list, labels=CLASS_NAMES, autopct='%1.1f%%', colors=COLORS, startangle=140)
ax2.set_title('Class Distribution', fontsize=14, fontweight='bold')

plt.suptitle('Dataset Overview — Original Images Only', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── Sample images grid ──────────────────────────────────────
n_cls = len(CLASS_NAMES)
fig, axes = plt.subplots(n_cls, 4, figsize=(14, n_cls * 2.8))
fig.suptitle('Sample Images per Class', fontsize=15, fontweight='bold', y=1.01)
import random; random.seed(SEED)

for row, cls_name in enumerate(CLASS_NAMES):
    cls_idx  = CLASS_NAMES.index(cls_name)
    cls_paths = [p for p, l in zip(all_paths, all_labels) if l == cls_idx]
    sample   = random.sample(cls_paths, min(4, len(cls_paths)))
    for col, img_path in enumerate(sample):
        try:
            img = Image.open(img_path).resize((IMG_SIZE, IMG_SIZE))
            axes[row][col].imshow(img)
        except Exception:
            pass
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(cls_name, fontsize=10, fontweight='bold', loc='left')

plt.tight_layout(); plt.show()


## ✂️ Stratified Train / Val / Test Split

Split is done **at file level before any augmentation**.
- `70%` train → gets augmentation inside the model
- `15%` val → used only for EarlyStopping / checkpoint selection
- `15%` test → **locked away**, used only once for final metrics

The test image list is saved to disk so results are always reproducible.

In [ ]:
# 70 / 15 / 15 stratified split
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.30, stratify=all_labels, random_state=SEED
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels,
    test_size=0.50, stratify=temp_labels, random_state=SEED
)

# Ensure all paths are plain strings (not Path objects)
train_paths = [str(p) for p in train_paths]
val_paths   = [str(p) for p in val_paths]
test_paths  = [str(p) for p in test_paths]

# Persist the test set list — never re-run this cell to avoid leakage
with open(TEST_LIST_PATH, 'w') as f:
    json.dump({'test_paths': test_paths, 'test_labels': test_labels,
               'class_names': CLASS_NAMES}, f, indent=2)

print('=== Stratified Split ===')
print(f'  Train : {len(train_paths):>5}  ({len(train_paths)/len(all_paths)*100:.0f}%)')
print(f'  Val   : {len(val_paths):>5}  ({len(val_paths)/len(all_paths)*100:.0f}%)')
print(f'  Test  : {len(test_paths):>5}  ({len(test_paths)/len(all_paths)*100:.0f}%)')
print(f'  Test list saved → {TEST_LIST_PATH}')

from collections import Counter
print('\n  Per-class counts in each split:')
print(f"  {'Class':<25} {'Train':>6} {'Val':>6} {'Test':>6}")
tc_tr = Counter(train_labels); tc_v = Counter(val_labels); tc_te = Counter(test_labels)
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<25} {tc_tr.get(i,0):>6} {tc_v.get(i,0):>6} {tc_te.get(i,0):>6}")

STEPS_PER_EPOCH = len(train_paths) // BATCH_SIZE
print(f'\nSteps/epoch: {STEPS_PER_EPOCH}')


## 🔧 TF Data Pipeline

> **Note on `.cache()` placement:**
> `train_ds` does **not** use `.cache()` — the augmentation layers inside
> the model produce different random transforms each epoch, so caching
> would freeze them. `val_ds` and `test_ds` are static and safe to cache.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    """Read, decode, and resize one image to float32 [0, 255]."""
    raw = tf.io.read_file(path)
    img = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img, label


def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds


# Augmentation is inside the model (training=True branch only),
# so we do NOT apply it here in the pipeline.
train_ds = make_dataset(train_paths, train_labels, shuffle=True)
val_ds   = make_dataset(val_paths,   val_labels,   shuffle=False).cache()
test_ds  = make_dataset(test_paths,  test_labels,  shuffle=False).cache()

NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train batches : {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Val   batches : {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"Test  batches : {tf.data.experimental.cardinality(test_ds).numpy()}")


In [ ]:
# Class weights — compensate for small Mouth Ulcer / Tooth Discoloration
weights = compute_class_weight('balanced',
                               classes=np.arange(NUM_CLASSES),
                               y=train_labels)
CLASS_WEIGHTS = dict(enumerate(weights))

print('Class Weights (higher = rarer class gets more attention):')
tc_tr = Counter(train_labels)
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<25} {tc_tr.get(i,0):>5} train imgs  weight={CLASS_WEIGHTS[i]:.3f}')


## 🔧 Augmentation

`get_augmentation()` returns a **new instance** every call.
Sharing one augmentation object across multiple models causes
conflicts in the TensorFlow computation graph and can break model saving.

In [ ]:
def get_augmentation(name_suffix=''):
    """
    Returns a fresh augmentation Sequential each call.
    Must NOT be shared between models.

    RandomContrast + RandomBrightness: help distinguish Calculus
    (yellow deposits) from Gingivitis (red/swollen gums) by varying
    the color response the model must learn from.
    Vertical flip omitted: dental images have fixed anatomical orientation.
    """
    return Sequential([
        layers.RandomRotation(0.12),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomFlip('horizontal'),
        layers.RandomZoom((-0.15, 0.10)),
        layers.RandomContrast(0.20),
        layers.RandomBrightness(0.15),
    ], name=f'augmentation{name_suffix}')


# Preview
_aug_prev = get_augmentation('_preview')
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Augmentation Preview (Top: Original | Bottom: Augmented)',
             fontsize=14, fontweight='bold')
for imgs, lbls in train_ds.take(1):
    for i in range(5):
        axes[0][i].imshow(imgs[i].numpy().astype('uint8'))
        axes[0][i].set_title(CLASS_NAMES[lbls[i]], fontsize=9); axes[0][i].axis('off')
        aug = _aug_prev(tf.expand_dims(imgs[i], 0), training=True)
        axes[1][i].imshow(tf.clip_by_value(aug[0], 0, 255).numpy().astype('uint8'))
        axes[1][i].set_title('Augmented', fontsize=9); axes[1][i].axis('off')
plt.tight_layout(); plt.show()
print('✅ get_augmentation() ready — each model gets its own fresh instance')


## 🎯 Focal Loss

Standard Cross-Entropy treats all examples equally.
**Focal Loss** (Lin et al., 2017) down-weights easy examples
and focuses training on hard misclassifications.

This directly addresses the **Calculus vs Gingivitis** confusion:
both look similar visually (yellow deposits near red gum tissue),
so the model keeps making confident mistakes on them.
Focal Loss forces it to study those hard cases more.

γ = 2.0 is the standard value; label_smoothing = 0.1 prevents overconfidence.

In [ ]:
class SparseFocalLoss(tf.keras.losses.Loss):
    """
    Focal Loss for multi-class sparse integer labels.

    L_focal = -sum( (1 - p_t)^gamma * log(p_t) )

    Args:
        gamma (float): Focusing parameter. 0 = standard CE. 2 = standard focal.
        label_smoothing (float): Reduces overconfidence. 0.1 is typical.
    """
    def __init__(self, gamma=2.0, label_smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        y_pred = tf.cast(y_pred, tf.float32)
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        batch  = tf.shape(y_pred)[0]

        y_pred_c = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        # Probability of the true class for each sample
        indices = tf.stack([tf.range(batch), y_true], axis=1)
        p_t     = tf.gather_nd(y_pred_c, indices)        # (batch,)

        # Focal weight
        focal_w = tf.pow(1.0 - p_t, self.gamma)

        # Cross-entropy for true class
        ce = -tf.math.log(p_t)

        # Label smoothing floor: adds uniform penalty to push away from extremes
        if self.label_smoothing > 0:
            n_cls     = tf.cast(tf.shape(y_pred)[-1], tf.float32)
            smooth_ce = -tf.reduce_sum(tf.math.log(y_pred_c), axis=-1) / n_cls
            ce = (1.0 - self.label_smoothing) * ce + self.label_smoothing * smooth_ce

        return tf.reduce_mean(focal_w * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'label_smoothing': self.label_smoothing})
        return cfg


FOCAL_LOSS = SparseFocalLoss(gamma=2.0, label_smoothing=0.1)
print('✅ SparseFocalLoss(gamma=2.0, label_smoothing=0.1) ready')
print('   Focuses training on hard Calculus / Gingivitis misclassifications')


## 🛠️ Helper Functions (All Fixes Applied)

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Shared helper functions
# ═══════════════════════════════════════════════════════════

def make_optimizer(steps_per_epoch, epochs, lr_max=1e-3):
    """Adam with CosineDecay schedule."""
    schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=lr_max,
        decay_steps=max(int(steps_per_epoch) * epochs, 1),
        alpha=0.01
    )
    return optimizers.Adam(schedule)


def get_callbacks(checkpoint):
    """Using EarlyStopping to stop training if my model stops improving, and saving the best one."""
    return [
        callbacks.EarlyStopping(monitor='val_accuracy', patience=30,
                                restore_best_weights=True, verbose=1),
        callbacks.ModelCheckpoint(checkpoint, save_best_only=True,
                                  monitor='val_accuracy', mode='max', verbose=0),
    ]


def train_and_time(model, name, epochs, checkpoint):
    """Function to train my model"""
    cbs = get_callbacks(checkpoint)
    print(f"\n{'━'*60}")
    print(f'  Training: {name}  ({epochs} epochs max)')
    print(f"{'━'*60}")
    t0   = time.time()
    hist = model.fit(
        train_ds, validation_data=val_ds,
        callbacks=cbs, class_weight=CLASS_WEIGHTS,
        epochs=epochs, verbose=1
    )
    elapsed = time.time() - t0
    print(f'⏱  {name}: {elapsed:.0f}s ({elapsed/60:.1f} min)')
    return hist, elapsed


def finetune(model, name, unfreeze_n, epochs=50, ckpt='ft.keras'):
    """
    Unfreeze last N layers and retrain with a low-LR CosineDecay.

    CRITICAL FIX: All BatchNormalization layers are kept frozen.
    Unfreezing BN during fine-tuning corrupts the running mean/variance
    statistics learned on ImageNet, destroying the pretrained features
    and causing accuracy to degrade instead of improve.
    """
    print(f'\n🔓 Fine-tuning {name} — last {unfreeze_n} layers unfrozen...')
    with strategy.scope():
        model.trainable = True
        # Freeze all layers before the last unfreeze_n
        for layer in model.layers[:len(model.layers) - unfreeze_n]:
            layer.trainable = False
        # ✅ CRITICAL: keep ALL BatchNorm layers frozen regardless
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = False
        model.compile(
            optimizer=make_optimizer(STEPS_PER_EPOCH, epochs, lr_max=1e-4),
            loss=FOCAL_LOSS,
            metrics=['accuracy']
        )
    trainable_n = sum(1 for l in model.layers if l.trainable)
    frozen_bn   = sum(1 for l in model.layers
                      if isinstance(l, tf.keras.layers.BatchNormalization))
    print(f'   Trainable layers : {trainable_n}/{len(model.layers)}')
    print(f'   Frozen BN layers : {frozen_bn}  ✅ pretrained stats protected')
    return train_and_time(model, f'{name} (Fine-tune)', epochs, ckpt)


def full_evaluate(model, name, dataset=None):
    """
    Evaluate on held-out test set.

    FIXES vs V1:
    1. Uses test_ds (never seen during training), not val_ds.
    2. model.predict(dataset) called ONCE — not per-batch in a loop.
       The loop pattern adds ~10x overhead from Python/TF graph switching.
    3. Single-image timing uses model(img, training=False) with a warm-up
       call to exclude JIT compilation cost from the benchmark.
    4. Returns y_true/y_pred — the original notebook omitted these,
       causing a KeyError crash in confusion-matrix cells.
    """
    if dataset is None:
        dataset = test_ds

    # ── Full-dataset prediction (single call) ────────────────
    preds  = model.predict(dataset, verbose=1)
    y_pred = np.argmax(preds, axis=1).tolist()
    y_true = []
    for _, lbls in dataset:
        y_true.extend(lbls.numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='weighted')

    # ── Real single-image inference timing ──────────────────
    sample = next(iter(dataset))[0][:20]
    _ = model(tf.expand_dims(sample[0], 0), training=False)  # JIT warm-up
    t0 = time.time()
    for img in sample:
        _ = model(tf.expand_dims(img, 0), training=False)
    inf_ms = (time.time() - t0) / len(sample) * 1000

    # ── Model size ───────────────────────────────────────────
    tmp = os.path.join(tempfile.gettempdir(), '_tmp_model_v2.keras')
    model.save(tmp)
    size_mb = os.path.getsize(tmp) / (1024 * 1024)
    os.remove(tmp)

    params = model.count_params()

    print(f"\n{'━'*60}")
    print(f'  {name} — Results on held-out test set')
    print(f"{'━'*60}")
    print(f'  Accuracy      : {acc:.4f}')
    print(f'  F1-Score      : {f1:.4f}')
    print(f'  Size          : {size_mb:.1f} MB')
    print(f'  Inference     : {inf_ms:.1f} ms/image  (real, after JIT warm-up)')
    print(f'  Parameters    : {params:,}')
    print(f'\n{classification_report(y_true, y_pred, target_names=CLASS_NAMES)}')

    return {
        'accuracy': acc, 'f1_score': f1, 'size_mb': size_mb,
        'inference_ms': inf_ms, 'train_time': 0, 'total_params': params,
        'y_true': y_true, 'y_pred': y_pred,
        'cm': confusion_matrix(y_true, y_pred).tolist(),
        'class_report': classification_report(
            y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
    }


def plot_curves(history, name):
    """Plot training curves; return final overfitting gap."""
    ta, va = history.history['accuracy'], history.history['val_accuracy']
    tl, vl = history.history['loss'],     history.history['val_loss']
    E = range(1, len(ta)+1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(E, tl, '#e94560', ls='--', lw=1.5, label='Train')
    ax1.plot(E, vl, '#4ecdc4', lw=2,   label='Val')
    b = np.argmin(vl)
    ax1.scatter(b+1, vl[b], s=120, c='#f7dc6f', zorder=5, edgecolors='k',
                label=f'Best: {vl[b]:.4f} @E{b+1}')
    ax1.set_title('Loss', fontweight='bold'); ax1.legend(); ax1.grid(alpha=0.2)
    ax2.plot(E, ta, '#e94560', ls='--', lw=1.5, label='Train')
    ax2.plot(E, va, '#4ecdc4', lw=2,   label='Val')
    b = np.argmax(va)
    ax2.scatter(b+1, va[b], s=120, c='#f7dc6f', zorder=5, edgecolors='k',
                label=f'Best: {va[b]:.4f} @E{b+1}')
    ax2.set_title('Accuracy', fontweight='bold'); ax2.legend(); ax2.grid(alpha=0.2)
    plt.suptitle(f'{name} — Training Curves', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
    return ta[-1] - va[-1]


def plot_cm(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=.5, linecolor='white')
    plt.title(f'{name} — Confusion Matrix', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.xticks(rotation=30); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()


def show_predictions(model, name, n=9):
    plt.figure(figsize=(13, 13))
    for imgs, lbls in test_ds.take(1):
        preds = np.argmax(model.predict(imgs, verbose=0), axis=1)
        for i in range(min(n, len(imgs))):
            ax = plt.subplot(3, 3, i+1)
            actual    = CLASS_NAMES[lbls[i]]
            predicted = CLASS_NAMES[preds[i]]
            ok = actual == predicted
            plt.imshow(imgs[i].numpy().astype('uint8'))
            title = (f"{'✓' if ok else '✗'} {predicted}" +
                     ('' if ok else f'\n(Actual: {actual})'))
            plt.title(title, color='#27ae60' if ok else '#e74c3c',
                      fontsize=10, fontweight='bold')
            plt.axis('off')
    plt.suptitle(f'{name} — Sample Predictions', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()


RESULTS = {}
OVERFIT  = {}
print('✅ All helper functions loaded!')


## 🧠 Model 1 — Custom CNN

In [ ]:
def build_cnn():
    """
    Custom CNN improvements over V1:
    - SpatialDropout2D after conv blocks (better regularization for spatial features)
    - get_augmentation() called fresh — not a shared global object
    - float32 final Dense for mixed-precision stability
    - Focal Loss instead of plain CrossEntropy
    """
    L2  = regularizers.l2(1e-4)
    aug = get_augmentation('_cnn')          # ✅ fresh, not shared

    m = Sequential([
        Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        aug,
        layers.Rescaling(1./255),

        Conv2D(64, 3, activation='relu', padding='same', kernel_regularizer=L2),
        BatchNormalization(),
        Conv2D(64, 3, activation='relu', padding='same', kernel_regularizer=L2),
        BatchNormalization(), MaxPooling2D(),
        SpatialDropout2D(0.10),             # ✅ spatial dropout for conv blocks

        Conv2D(128, 3, activation='relu', padding='same', kernel_regularizer=L2),
        BatchNormalization(),
        Conv2D(128, 3, activation='relu', padding='same', kernel_regularizer=L2),
        BatchNormalization(), MaxPooling2D(),
        SpatialDropout2D(0.10),

        Conv2D(256, 3, activation='relu', padding='same', kernel_regularizer=L2),
        BatchNormalization(),
        Conv2D(256, 3, activation='relu', padding='same', kernel_regularizer=L2),
        BatchNormalization(), MaxPooling2D(),
        SpatialDropout2D(0.15),

        Conv2D(512, 3, activation='relu', padding='same', kernel_regularizer=L2),
        BatchNormalization(), MaxPooling2D(),

        GlobalAveragePooling2D(),
        Dense(512, activation='relu', kernel_regularizer=L2),
        BatchNormalization(), Dropout(0.50),
        Dense(256, activation='relu', kernel_regularizer=L2), Dropout(0.40),
        Dense(NUM_CLASSES, activation='softmax', dtype='float32'),  # float32 for stability
    ], name='Custom_CNN')

    m.compile(
        optimizer=make_optimizer(STEPS_PER_EPOCH, EPOCHS, lr_max=1e-3),
        loss=FOCAL_LOSS,
        metrics=['accuracy']
    )
    return m

with strategy.scope():
    cnn = build_cnn()
cnn.summary()


In [ ]:
cnn_hist, cnn_time = train_and_time(cnn, 'Custom CNN', EPOCHS, 'best_cnn.keras')


In [ ]:
overfit_cnn = plot_curves(cnn_hist, 'Custom CNN')
OVERFIT['Custom CNN'] = overfit_cnn
show_predictions(cnn, 'Custom CNN')
res_cnn = full_evaluate(cnn, 'Custom CNN')
res_cnn['train_time'] = cnn_time
RESULTS['Custom CNN'] = res_cnn
plot_cm(res_cnn['y_true'], res_cnn['y_pred'], 'Custom CNN')
cnn.save('final_cnn.keras')
print('💾 Saved: final_cnn.keras')
import json
with open('metrics.json', 'w') as f:
    json.dump({k: {'accuracy': v['accuracy'], 'f1_score': v['f1_score'], 'size_mb': v['size_mb'], 'inference_ms': v['inference_ms'], 'train_time': v.get('train_time', 0)} for k, v in RESULTS.items()}, f, indent=4)
print('📊 Metrics updated in metrics.json')


## 🧠 Model 2 — EfficientNetB0 (Transfer Learning)

In [ ]:
def build_pretrained(base_class, preprocess_fn, name):
    """
    Build pretrained transfer-learning model.
    Each model gets its own fresh augmentation instance (no sharing).
    float32 output for mixed-precision stability.
    """
    aug    = get_augmentation(f'_{name}')   # ✅ fresh per model
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x      = aug(inputs)

    if preprocess_fn is not None:
        x = preprocess_fn(x)

    base = base_class(include_top=False, weights='imagenet', input_tensor=x)
    base.trainable = False

    x   = GlobalAveragePooling2D()(base.output)
    x   = BatchNormalization()(x)
    x   = Dropout(0.4)(x)
    x   = Dense(256, activation='relu')(x)
    x   = BatchNormalization()(x)
    x   = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    return tf.keras.Model(inputs, out, name=name)

print('✅ build_pretrained() ready')


In [ ]:
with strategy.scope():
    eff = build_pretrained(EfficientNetB0, None, 'EfficientNetB0')
    eff.compile(
        optimizer=make_optimizer(STEPS_PER_EPOCH, PHASE1_EPOCHS, lr_max=1e-3),
        loss=FOCAL_LOSS,
        metrics=['accuracy']
    )


In [ ]:
# Phase 1: frozen base warm-up
eff_h1, eff_t1 = train_and_time(eff, 'EfficientNetB0 (Frozen)', PHASE1_EPOCHS, 'best_eff_p1.keras')
# Phase 2: fine-tune last 37 layers (BN stays frozen — see finetune() docstring)
eff_h2, eff_t2 = finetune(eff, 'EfficientNetB0', 37, epochs=FINETUNE_EPOCHS, ckpt='best_eff_ft.keras')


In [ ]:
overfit_eff = plot_curves(eff_h2, 'EfficientNetB0')
OVERFIT['EfficientNetB0'] = overfit_eff
show_predictions(eff, 'EfficientNetB0')
res_eff = full_evaluate(eff, 'EfficientNetB0')
res_eff['train_time'] = eff_t1 + eff_t2
RESULTS['EfficientNetB0'] = res_eff
plot_cm(res_eff['y_true'], res_eff['y_pred'], 'EfficientNetB0')
eff.save('final_EfficientNetB0.keras')
print('💾 Saved: final_EfficientNetB0.keras')
import json
with open('metrics.json', 'w') as f:
    json.dump({k: {'accuracy': v['accuracy'], 'f1_score': v['f1_score'], 'size_mb': v['size_mb'], 'inference_ms': v['inference_ms'], 'train_time': v.get('train_time', 0)} for k, v in RESULTS.items()}, f, indent=4)
print('📊 Metrics updated in metrics.json')


## 🧠 Model 3 — ResNet50

In [ ]:
with strategy.scope():
    res_model = build_pretrained(ResNet50, resnet.preprocess_input, 'ResNet50')
    res_model.compile(
        optimizer=make_optimizer(STEPS_PER_EPOCH, PHASE1_EPOCHS, lr_max=1e-3),
        loss=FOCAL_LOSS,
        metrics=['accuracy']
    )


In [ ]:
res_h1, res_t1 = train_and_time(res_model, 'ResNet50 (Frozen)', PHASE1_EPOCHS, 'best_res_p1.keras')
res_h2, res_t2 = finetune(res_model, 'ResNet50', 47, epochs=FINETUNE_EPOCHS, ckpt='best_res_ft.keras')


In [ ]:
overfit_res = plot_curves(res_h2, 'ResNet50')
OVERFIT['ResNet50'] = overfit_res
show_predictions(res_model, 'ResNet50')
res_eval = full_evaluate(res_model, 'ResNet50')
res_eval['train_time'] = res_t1 + res_t2
RESULTS['ResNet50'] = res_eval
plot_cm(res_eval['y_true'], res_eval['y_pred'], 'ResNet50')
res_model.save('final_ResNet50.keras')
print('💾 Saved: final_ResNet50.keras')
import json
with open('metrics.json', 'w') as f:
    json.dump({k: {'accuracy': v['accuracy'], 'f1_score': v['f1_score'], 'size_mb': v['size_mb'], 'inference_ms': v['inference_ms'], 'train_time': v.get('train_time', 0)} for k, v in RESULTS.items()}, f, indent=4)
print('📊 Metrics updated in metrics.json')


## 🧠 Model 4 — DenseNet121

In [ ]:
with strategy.scope():
    dense_model = build_pretrained(DenseNet121, densenet.preprocess_input, 'DenseNet121')
    dense_model.compile(
        optimizer=make_optimizer(STEPS_PER_EPOCH, PHASE1_EPOCHS, lr_max=1e-3),
        loss=FOCAL_LOSS,
        metrics=['accuracy']
    )


In [ ]:
dense_h1, dense_t1 = train_and_time(dense_model, 'DenseNet121 (Frozen)', PHASE1_EPOCHS, 'best_dense_p1.keras')
dense_h2, dense_t2 = finetune(dense_model, 'DenseNet121', 47, epochs=FINETUNE_EPOCHS, ckpt='best_dense_ft.keras')


In [ ]:
overfit_dense = plot_curves(dense_h2, 'DenseNet121')
OVERFIT['DenseNet121'] = overfit_dense
show_predictions(dense_model, 'DenseNet121')
res_dense = full_evaluate(dense_model, 'DenseNet121')
res_dense['train_time'] = dense_t1 + dense_t2
RESULTS['DenseNet121'] = res_dense
plot_cm(res_dense['y_true'], res_dense['y_pred'], 'DenseNet121')
dense_model.save('final_DenseNet121.keras')
print('💾 Saved: final_DenseNet121.keras')
import json
with open('metrics.json', 'w') as f:
    json.dump({k: {'accuracy': v['accuracy'], 'f1_score': v['f1_score'], 'size_mb': v['size_mb'], 'inference_ms': v['inference_ms'], 'train_time': v.get('train_time', 0)} for k, v in RESULTS.items()}, f, indent=4)
print('📊 Metrics updated in metrics.json')


## 💾 Save Models & Metrics

In [ ]:

best_name = max(RESULTS, key=lambda x: RESULTS[x]['accuracy'])

export_results = {}
for name, r in RESULTS.items():
    export_results[name] = {
        'accuracy':     r['accuracy'],
        'f1_score':     r['f1_score'],
        'size_mb':      r['size_mb'],
        'inference_ms': r['inference_ms'],
        'train_time':   r['train_time'],
        'total_params': r['total_params'],
        'cm':           r.get('cm'),
        'class_report': r.get('class_report'),
    }

HISTORIES = {
    'Custom CNN':     {k: [float(v) for v in cnn_hist.history[k]]   for k in cnn_hist.history},
    'EfficientNetB0': {k: [float(v) for v in eff_h2.history[k]]    for k in eff_h2.history},
    'ResNet50':       {k: [float(v) for v in res_h2.history[k]]    for k in res_h2.history},
    'DenseNet121':    {k: [float(v) for v in dense_h2.history[k]]  for k in dense_h2.history},
}

export_data = {
    'HISTORIES':    HISTORIES,
    'RESULTS':      export_results,
    'OVERFIT':      OVERFIT,
    'best_name':    best_name,
    'CLASS_NAMES':  CLASS_NAMES,
    'class_counts': class_counts_orig,
    'split_info': {
        'train': len(train_paths),
        'val':   len(val_paths),
        'test':  len(test_paths),
        'note':  'Stratified 70/15/15 on original images only. Zero data leakage.'
    }
}

with open('training_metrics.json', 'w') as f:
    json.dump(export_data, f, indent=4)

print('✅ All 4 models saved (.keras)')
print('✅ Metrics saved (training_metrics.json)')
print(f'\n🏆 Best model: {best_name}  (Accuracy: {RESULTS[best_name]["accuracy"]:.4f})')


In [ ]:
import zipfile
from IPython.display import FileLink, display

zip_name    = 'Oral_Disease_Models_Final.zip'
files_to_zip = [
    'final_cnn.keras',
    'final_EfficientNetB0.keras',
    'final_ResNet50.keras',
    'final_DenseNet121.keras',
    'training_metrics.json',
    'test_image_list.json',
]

with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for file in files_to_zip:
        if os.path.exists(file):
            zf.write(file)
            print(f'  ✔ {file}')
        else:
            print(f'  ✗ {file} not found (skipped)')

print(f'\n📦 Packaged → {zip_name}')
print('⬇️  Click to download:')
display(FileLink(zip_name))
